# Zirconia O₂ model: run the app

Run the cells from top to bottom. Keep this notebook in the same folder as `app.py`.

1. Check the requirements
2. Start the server
3. Open the UI (link, or embedded below)
4. Stop the server when you are done

In [ ]:
# 1. Check the requirements (flask + numpy only; no internet needed)
import importlib, os, sys
for pkg in ("flask", "numpy"):
    try:
        importlib.import_module(pkg); print(f"OK   {pkg}")
    except ImportError:
        print(f"MISSING {pkg}  ->  pip install {pkg}")
for f in ("app.py", "model.py", "templates/index.html", "static/app.js", "static/style.css",
          "zirconia_sensor_experimental_data.xlsx"):
    print("OK  " if os.path.exists(f) else "MISSING", f)

In [ ]:
# 2. Start the server in the background (re-running this cell restarts it)
import subprocess, sys, time, urllib.request

HOST, PORT = "127.0.0.1", 5050      # use HOST = "0.0.0.0" if Jupyter runs on a remote machine
URL = f"http://127.0.0.1:{PORT}"

if "proc" in globals() and proc.poll() is None:
    proc.terminate(); proc.wait()

env = {**os.environ, "HOST": HOST, "PORT": str(PORT)}
log = open("app.log", "w")
proc = subprocess.Popen([sys.executable, "app.py"], stdout=log, stderr=subprocess.STDOUT, env=env)

for _ in range(60):
    try:
        urllib.request.urlopen(URL + "/api/results", timeout=1); break
    except Exception:
        if proc.poll() is not None: break
        time.sleep(0.5)

print(open("app.log").read())
print("Running:" if proc.poll() is None else "FAILED, see the log above:", URL)

In [ ]:
# 3. Open the UI: click the link, or view it embedded right here
from IPython.display import IFrame, HTML, display
display(HTML(f'<a href="{URL}" target="_blank">Open {URL} in a new tab</a>'))
IFrame(URL, width="100%", height=800)

In [ ]:
# Optional: predict straight from Python
import json
P, I = 56.89, 8.082
r = json.load(urllib.request.urlopen(f"{URL}/api/predict?P={P}&I={I}"))
print(f"P={P}, I={I} uA  ->  C = {r['C_corr']:.2f} %   (physics only {r['C_phys']:.2f} %)")

In [ ]:
# 4. Stop the server
proc.terminate(); proc.wait(); print("stopped")